### Import Libraries


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, recall_score, precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

### Load Data
load `dataset.csv` into file section


In [ ]:
df = pd.read_csv("dataset.csv")
df.head()

,date,MaxTemp,MinTemp,MeanTemp,TotalRainfall,Relativehumidity00am,Surfacepressure00am,Windspeed00am,Cloud00am,Relativehumidity03am,...,Relativehumidity06pm,Surfacepressure06pm,Windspeed06pm,Cloud06pm,Relativehumidity09pm,Surfacepressure09pm,Windspeed09pm,Cloud09pm,HeavyRainTomorrow,Rain10Tomorrow
0,2015-01-01,-4.036,-8.486000,-6.692250,0.0,35.259354,1018.76770,21.407139,0.0,32.687237,...,38.985610,1021.46893,14.512064,2.0,48.663970,1022.8485,12.904882,5.0,0,0
1,2015-01-02,-1.686,-7.486000,-5.196417,0.0,50.996235,1022.55200,11.720751,94.0,51.067000,...,32.183067,1022.58250,10.009036,0.0,38.726048,1023.8568,8.714677,0.0,0,0
2,2015-01-03,1.864,-8.535999,-3.625583,0.0,45.030655,1023.84015,6.193674,0.0,44.715717,...,72.227960,1018.03810,7.993298,83.0,75.274864,1017.1386,5.600286,100.0,0,0
3,2015-01-04,7.364,-2.136000,1.684833,0.0,79.950970,1016.14014,6.989936,72.0,80.911354,...,90.202210,1014.90690,10.028439,22.0,97.528824,1016.2873,4.394360,7.0,0,0
4,2015-01-05,8.314,-2.336000,2.257750,1.0,99.266430,1016.04175,5.804825,100.0,98.177280,...,61.907803,1011.45245,5.191994,100.0,77.188675,1009.3218,1.138420,100.0,0,0


### Feature Selection and Data Cleaning


In [ ]:
features = [
    "MaxTemp", "MinTemp", "MeanTemp", "TotalRainfall", "Relativehumidity00am", "Surfacepressure00am", "Windspeed00am", "Cloud00am", "Relativehumidity03am", "Surfacepressure03am", "Windspeed03am", "Cloud03am", "Relativehumidity06am", "Surfacepressure06am", "Windspeed06am", "Cloud06am", "Relativehumidity09am", "Surfacepressure09am", "Windspeed09am", "Cloud09am", "Relativehumidity00pm", "Surfacepressure00pm", "Windspeed00pm", "Cloud00pm", "Relativehumidity03pm", "Surfacepressure03pm", "Windspeed03pm", "Cloud03pm", "Relativehumidity06pm", "Surfacepressure06pm", "Windspeed06pm", "Cloud06pm", "Relativehumidity09pm", "Surfacepressure09pm", "Windspeed09pm", "Cloud09pm"
]

data = df[features + ["Rain10Tomorrow"]].dropna()

### Data Splitting


In [ ]:
X = data[features].values
y = data["Rain10Tomorrow"].values

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

### Feature Scaling


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

NameError: name 'StandardScaler' is not defined

### Isolate Minority Class Samples


In [ ]:
X_heavy = X_train_scaled[y_train == 1]

print("Heavy-rain samples:", X_heavy.shape)

Heavy-rain samples: (2258, 36)


### VAE Model Definition


In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()

        #encoder
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        #decoder
        self.fc1_dec = nn.Linear(latent_dim, hidden_dim)
        self.fc2_dec = nn.Linear(hidden_dim, input_dim)

        # activation fnction definition
        self.silu = nn.SiLU()
        self.relu = nn.ReLU()
        self.prelu = nn.PReLU()


    def encode(self, x):
        h = self.silu(self.fc1(x)) # change the function here
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z

    def decode(self, z):
        h = self.fc1_dec(z)
        x_recon = self.fc2_dec(h)
        return x_recon

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar

### VAE Loss Function


In [ ]:
def vae_loss(x_recon, x, mu, logvar):
    recon_loss = nn.functional.mse_loss(x_recon, x, reduction="mean")
    kl_loss = 0.5*torch.sum(mu**2 + logvar.exp() - logvar - 1)
    return recon_loss + 0.0001 * kl_loss

### Initialize VAE and Optimizer


In [ ]:
input_dim = X_train_scaled.shape[1]
hidden_dim = 18
latent_dim = 4

vae = VAE(input_dim=input_dim, hidden_dim=hidden_dim, latent_dim=latent_dim)

optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

### Train the VAE


In [ ]:
X_heavy_tensor = torch.tensor(X_heavy, dtype=torch.float32)

dataset = TensorDataset(X_heavy_tensor)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

epochs = 2000

vae.train()

for epoch in range(epochs):
    total_loss = 0

    for batch in loader:
        x_batch = batch[0]

        optimizer.zero_grad()

        x_recon, mu, logvar = vae(x_batch)
        loss = vae_loss(x_recon, x_batch, mu, logvar)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

Epoch 100/2000, Loss: 8.5815
Epoch 200/2000, Loss: 8.5896
Epoch 300/2000, Loss: 8.5805
Epoch 400/2000, Loss: 8.5774
Epoch 500/2000, Loss: 8.5657
Epoch 600/2000, Loss: 8.6359
Epoch 700/2000, Loss: 8.5663
Epoch 800/2000, Loss: 8.5454
Epoch 900/2000, Loss: 8.5372
Epoch 1000/2000, Loss: 8.5989
Epoch 1100/2000, Loss: 8.6121
Epoch 1200/2000, Loss: 8.6073
Epoch 1300/2000, Loss: 8.6189
Epoch 1400/2000, Loss: 8.5054
Epoch 1500/2000, Loss: 8.5236
Epoch 1600/2000, Loss: 8.5137
Epoch 1700/2000, Loss: 8.5441
Epoch 1800/2000, Loss: 8.5767
Epoch 1900/2000, Loss: 8.5381
Epoch 2000/2000, Loss: 8.5397


### Generate Synthetic Samples


In [ ]:
vae.eval()

# Generate fewer synthetic samples: only 50% of original heavy-rain samples
n_generate = int(len(X_heavy) * 0.5)

with torch.no_grad():
    z = torch.randn(n_generate, latent_dim)
    synthetic_heavy = vae.decode(z).numpy()

synthetic_labels = np.ones(n_generate)

print("Generated synthetic samples:", synthetic_heavy.shape)

Generated synthetic samples: (1129, 36)


### Implementing GMM in Latent Space

In [ ]:
from sklearn.mixture import GaussianMixture

# Get latent representations for the heavy-rain samples
vae.eval() # Set VAE to evaluation mode
with torch.no_grad():
    mu_heavy, logvar_heavy = vae.encode(X_heavy_tensor)
    latent_heavy = vae.reparameterize(mu_heavy, logvar_heavy).numpy()

print(f"Latent representations shape: {latent_heavy.shape}")

Latent representations shape: (2258, 4)


In [ ]:
n_components = 5 # Number of Gaussian components
gmm = GaussianMixture(n_components=n_components, random_state=42)
gmm.fit(latent_heavy)

print("GMM fitted successfully to the latent space.")

GMM fitted successfully to the latent space.


In [ ]:
# Generate new latent vectors from the GMM
n_generate_gmm = int(len(X_heavy) * 0.5) # Generate the same number of samples as before
latent_gmm_samples, _ = gmm.sample(n_generate_gmm)
latent_gmm_samples_tensor = torch.tensor(latent_gmm_samples, dtype=torch.float32)

# Decode these latent samples back to the original feature space
with torch.no_grad():
    synthetic_heavy_gmm = vae.decode(latent_gmm_samples_tensor).numpy()

synthetic_labels_gmm = np.ones(n_generate_gmm)

print(f"Generated synthetic samples using GMM: {synthetic_heavy_gmm.shape}")

Generated synthetic samples using GMM: (1129, 36)


### VAE-GMM Augmented Logistic Regression Model

In [ ]:
X_train_aug_gmm = np.vstack([X_train_scaled, synthetic_heavy_gmm])
y_train_aug_gmm = np.concatenate([y_train, synthetic_labels_gmm])

proposed_gmm_lr = LogisticRegression(max_iter=1000, class_weight="balanced")
proposed_gmm_lr.fit(X_train_aug_gmm, y_train_aug_gmm)

y_pred_aug_gmm_lr = proposed_gmm_lr.predict(X_test_scaled)

print("VAE-GMM Augmented Logistic Regression")
print(classification_report(y_test, y_pred_aug_gmm_lr))
print(confusion_matrix(y_test, y_pred_aug_gmm_lr))

VAE-GMM Augmented Logistic Regression
              precision    recall  f1-score   support

           0       0.97      0.65      0.78      5062
           1       0.21      0.83      0.33       564

    accuracy                           0.67      5626
   macro avg       0.59      0.74      0.56      5626
weighted avg       0.90      0.67      0.73      5626

[[3285 1777]
 [  96  468]]


### VAE-GMM Augmented Random Forest Model

In [ ]:
# Ensure X_train_aug_gmm and y_train_aug_gmm are defined for Random Forest if not already run in the LR section.
X_train_aug_gmm = np.vstack([X_train_scaled, synthetic_heavy_gmm])
y_train_aug_gmm = np.concatenate([y_train, synthetic_labels_gmm])

proposed_gmm_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

proposed_gmm_rf.fit(X_train_aug_gmm, y_train_aug_gmm)
y_pred_aug_gmm_rf = proposed_gmm_rf.predict(X_test_scaled)

print("VAE-GMM Augmented Random Forest")
print(classification_report(y_test, y_pred_aug_gmm_rf))
print(confusion_matrix(y_test, y_pred_aug_gmm_rf))

VAE-GMM Augmented Random Forest
              precision    recall  f1-score   support

           0       0.91      0.99      0.95      5062
           1       0.61      0.12      0.20       564

    accuracy                           0.90      5626
   macro avg       0.76      0.56      0.57      5626
weighted avg       0.88      0.90      0.87      5626

[[5019   43]
 [ 497   67]]


### Baseline Logistic Regression Model


In [ ]:
baseline = LogisticRegression(max_iter=1000, class_weight="balanced")
baseline.fit(X_train_scaled, y_train)

y_pred_base = baseline.predict(X_test_scaled)

print(classification_report(y_test, y_pred_base))
print(confusion_matrix(y_test, y_pred_base))

              precision    recall  f1-score   support

           0       0.97      0.64      0.77      5062
           1       0.21      0.84      0.33       564

    accuracy                           0.66      5626
   macro avg       0.59      0.74      0.55      5626
weighted avg       0.90      0.66      0.73      5626

[[3250 1812]
 [  93  471]]


### VAE-Augmented Logistic Regression Model


In [ ]:
X_train_aug = np.vstack([X_train_scaled, synthetic_heavy])
y_train_aug = np.concatenate([y_train, synthetic_labels])

proposed = LogisticRegression(max_iter=1000, class_weight="balanced")
proposed.fit(X_train_aug, y_train_aug)

y_pred_aug = proposed.predict(X_test_scaled)

print(classification_report(y_test, y_pred_aug))
print(confusion_matrix(y_test, y_pred_aug))

              precision    recall  f1-score   support

           0       0.97      0.64      0.77      5062
           1       0.21      0.84      0.33       564

    accuracy                           0.66      5626
   macro avg       0.59      0.74      0.55      5626
weighted avg       0.90      0.66      0.73      5626

[[3221 1841]
 [  88  476]]


### Baseline Random Forest Model


In [ ]:
baseline_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

baseline_rf.fit(X_train_scaled, y_train)
y_pred_base_rf = baseline_rf.predict(X_test_scaled)

print("Baseline Random Forest")
print(classification_report(y_test, y_pred_base_rf))
print(confusion_matrix(y_test, y_pred_base_rf))

Baseline Random Forest
              precision    recall  f1-score   support

           0       0.91      0.99      0.95      5062
           1       0.64      0.10      0.18       564

    accuracy                           0.90      5626
   macro avg       0.78      0.55      0.56      5626
weighted avg       0.88      0.90      0.87      5626

[[5030   32]
 [ 506   58]]


### VAE-Augmented Random Forest Model




In [ ]:
proposed_rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

proposed_rf.fit(X_train_aug, y_train_aug)
y_pred_aug_rf = proposed_rf.predict(X_test_scaled)

print("VAE-Augmented Random Forest")
print(classification_report(y_test, y_pred_aug_rf))
print(confusion_matrix(y_test, y_pred_aug_rf))

VAE-Augmented Random Forest
              precision    recall  f1-score   support

           0       0.91      0.99      0.95      5062
           1       0.65      0.13      0.21       564

    accuracy                           0.91      5626
   macro avg       0.78      0.56      0.58      5626
weighted avg       0.88      0.91      0.88      5626

[[5024   38]
 [ 493   71]]


### Random Forest Results Comparison


In [ ]:
results_rf = pd.DataFrame({
    "Model": ["Baseline RF", "VAE-Augmented RF"],
    "Recall": [
        recall_score(y_test, y_pred_base_rf),
        recall_score(y_test, y_pred_aug_rf)
    ],
    "Precision": [
        precision_score(y_test, y_pred_base_rf),
        precision_score(y_test, y_pred_aug_rf)
    ],
    "F1-score": [
        f1_score(y_test, y_pred_base_rf),
        f1_score(y_test, y_pred_aug_rf)
    ]
})

results_rf

,Model,Recall,Precision,F1-score
0,Baseline RF,0.102837,0.644444,0.177370
1,VAE-Augmented RF,0.125887,0.651376,0.210996


### Cell `BaTr6uf05reM`: Overall Results Comparison


In [ ]:
all_results = pd.DataFrame({
    "Model": [
        "Baseline Logistic Regression",
        "VAE-Augmented Logistic Regression",
        "VAE-GMM Augmented Logistic Regression",
        "Baseline Random Forest",
        "VAE-Augmented Random Forest",
        "VAE-GMM Augmented Random Forest"
    ],
    "Recall": [
        recall_score(y_test, y_pred_base),
        recall_score(y_test, y_pred_aug),
        recall_score(y_test, y_pred_aug_gmm_lr),
        recall_score(y_test, y_pred_base_rf),
        recall_score(y_test, y_pred_aug_rf),
        recall_score(y_test, y_pred_aug_gmm_rf)
    ],
    "Precision": [
        precision_score(y_test, y_pred_base),
        precision_score(y_test, y_pred_aug),
        precision_score(y_test, y_pred_aug_gmm_lr),
        precision_score(y_test, y_pred_base_rf),
        precision_score(y_test, y_pred_aug_rf),
        precision_score(y_test, y_pred_aug_gmm_rf)
    ],
    "F1-score": [
        f1_score(y_test, y_pred_base),
        f1_score(y_test, y_pred_aug),
        f1_score(y_test, y_pred_aug_gmm_lr),
        f1_score(y_test, y_pred_base_rf),
        f1_score(y_test, y_pred_aug_rf),
        f1_score(y_test, y_pred_aug_gmm_rf)
    ]
})

display(all_results)

,Model,Recall,Precision,F1-score
0,Baseline Logistic Regression,0.835106,0.206307,0.330875
1,VAE-Augmented Logistic Regression,0.843972,0.205438,0.330441
2,VAE-GMM Augmented Logistic Regression,0.829787,0.208463,0.333215
3,Baseline Random Forest,0.102837,0.644444,0.177370
4,VAE-Augmented Random Forest,0.125887,0.651376,0.210996
5,VAE-GMM Augmented Random Forest,0.118794,0.609091,0.198813
